In [1]:
# %%
from pathlib import Path
import re
import json
import math
import hashlib
from collections import defaultdict

import pandas as pd
from tqdm import tqdm

import spacy
import scispacy  # noqa: F401  (ensures scispacy entry points are registered)
from scispacy.umls_linking import UmlsEntityLinker

import networkx as nx
import pickle

In [2]:
# %%
PROJECT_ROOT = Path("..").resolve()
PARSED_DIR = PROJECT_ROOT / "data" / "parsed"

IN_PARQUET = PARSED_DIR / "pmc_retrieval_candidates.parquet"

OUT_MENTIONS_RAW = PARSED_DIR / "graph_mentions_raw.parquet"
OUT_CONCEPTS     = PARSED_DIR / "graph_concepts.parquet"
OUT_MENTIONS     = PARSED_DIR / "graph_mentions.parquet"

OUT_EDGE_EVID    = PARSED_DIR / "graph_edge_evidence.parquet"
OUT_EDGE_SC      = PARSED_DIR / "graph_edges_symptom_condition.parquet"

OUT_NODES        = PARSED_DIR / "graph_nodes.parquet"
OUT_EDGES        = PARSED_DIR / "graph_edges.parquet"

OUT_NX_PKL       = PARSED_DIR / "graph_nx.pkl"

assert IN_PARQUET.exists(), f"Missing input: {IN_PARQUET}"

In [3]:
# %%
df = pd.read_parquet(IN_PARQUET)
print("Chunks:", len(df))
print("PMCIDs:", df["pmcid"].nunique())
df.head(3)

Chunks: 12935
PMCIDs: 814


,chunk_id,pmcid,article_title,section,chunk_index,chunk_text,token_count,license
0,9fd0d269d90d3d89b02c81308d3a1500b6b91c37,PMC12446057,The immune checkpoint TIM-3/HMGB-1 axis in myo...,discussion,0,"we investigated, for the first time, expressio...",350,CC BY
1,79b3541338632b7d1dd88f1b98e362958d8a7411,PMC12446057,The immune checkpoint TIM-3/HMGB-1 axis in myo...,discussion,1,"an mi. on pbmc level, overall tim - 3 rna expr...",350,CC BY
2,5020b51f114f3fc6edf8746b2206e74869294b06,PMC12446057,The immune checkpoint TIM-3/HMGB-1 axis in myo...,discussion,2,"to be more organ - bound, possibly to maintain...",350,CC BY


In [4]:
# %%
MODEL = "en_core_sci_sm"  # change to en_core_sci_md if installed and you want better quality

try:
    nlp = spacy.load(MODEL)
except Exception as e:
    raise RuntimeError(
        f"Failed to load spaCy model '{MODEL}'. Install it and retry. Original error: {e}"
    )

# Initialize UMLS linker object (provides KB + linking)
try:
    linker = UmlsEntityLinker(resolve_abbreviations=True)
except Exception as e:
    raise RuntimeError(
        "Failed to initialize UmlsEntityLinker. This usually means the SciSpaCy UMLS "
        "resources are not installed/available in this environment. "
        f"Original error: {e}"
    )

# Add SciSpaCy linker pipe via its REGISTERED spaCy factory name
if "scispacy_linker" not in nlp.pipe_names:
    nlp.add_pipe("scispacy_linker", last=True)

# Ensure the pipe uses the KB we initialized (defensive; usually already wired)
try:
    nlp.get_pipe("scispacy_linker").kb = linker.kb
except Exception:
    pass

print("Pipeline:", nlp.pipe_names)

c:\Users\aarav\anaconda3\envs\pmc_graphrag\lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]
c:\Users\aarav\anaconda3\envs\pmc_graphrag\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.1.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\aarav\anaconda3\envs\pmc_graphrag\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.1.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persisten

Pipeline: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner', 'scispacy_linker']


In [5]:
# %%
SYMPTOM_TUIS = {
    "T184",  # Sign or Symptom
    "T033",  # Finding
    "T034",  # Laboratory or Test Result
}

CONDITION_TUIS = {
    "T047",  # Disease or Syndrome
    "T048",  # Mental or Behavioral Dysfunction
    "T050",  # Experimental Model of Disease
    "T191",  # Neoplastic Process
    "T046",  # Pathologic Function
    "T190",  # Anatomical Abnormality
    # "T049",  # Cell or Molecular Dysfunction (optional; can add later if needed)
}

In [6]:
# %%
def clean_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

def sha1(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()

def make_mention_id(chunk_id: str, start: int, end: int, cui: str, mtype: str) -> str:
    return sha1(f"{chunk_id}|{start}|{end}|{cui}|{mtype}")

def get_kb_entity(cui: str):
    # scispacy linker kb lookup
    return linker.kb.cui_to_entity.get(cui)

In [ ]:
# %%
mentions_raw = []
concept_rows = {}  # concept_id -> dict

# For speed, use nlp.pipe batching
texts = df["chunk_text"].tolist()
chunk_ids = df["chunk_id"].tolist()
pmcids = df["pmcid"].tolist()
sections = df["section"].tolist()
titles = df["article_title"].tolist()

BATCH = 32

for i in tqdm(range(0, len(df), BATCH)):
    batch_texts = texts[i:i+BATCH]
    docs = list(nlp.pipe(batch_texts))

    for j, doc in enumerate(docs):
        idx = i + j
        chunk_id = chunk_ids[idx]
        pmcid = pmcids[idx]
        section = sections[idx]
        title = titles[idx]

        for ent in doc.ents:
            ent_text = clean_ws(ent.text)
            if not ent_text:
                continue

            kb_ents = getattr(ent._, "kb_ents", [])
            if not kb_ents:
                continue

            start = int(ent.start_char)
            end = int(ent.end_char)

            # keep top-N candidates per span
            for cui, score in kb_ents[:5]:
                kb_ent = get_kb_entity(cui)
                if kb_ent is None:
                    continue

                canonical = clean_ws(getattr(kb_ent, "canonical_name", "")) or None
                tuis = list(getattr(kb_ent, "types", []) or [])
                tuiset = set(tuis)

                is_symptom = len(tuiset.intersection(SYMPTOM_TUIS)) > 0
                is_condition = len(tuiset.intersection(CONDITION_TUIS)) > 0

                if not (is_symptom or is_condition):
                    continue

                concept_id = f"UMLS:{cui}"
                if concept_id not in concept_rows:
                    concept_rows[concept_id] = {
                        "concept_id": concept_id,
                        "cui": cui,
                        "canonical_name": canonical,
                        "tuis": tuis,
                        "source_vocab": "UMLS",
                        "is_grounded": True,
                    }

                if is_symptom:
                    mention_id = make_mention_id(chunk_id, start, end, cui, "SYMPTOM")
                    mentions_raw.append({
                        "mention_id": mention_id,
                        "chunk_id": chunk_id,
                        "pmcid": pmcid,
                        "article_title": title,
                        "section": section,
                        "start_char": start,
                        "end_char": end,
                        "mention_text": ent_text,
                        "mention_type": "SYMPTOM",
                        "concept_id": concept_id,
                        "link_score": float(score),
                        "detector": "SCISPACY_UMLS",
                    })

                if is_condition:
                    mention_id = make_mention_id(chunk_id, start, end, cui, "CONDITION")
                    mentions_raw.append({
                        "mention_id": mention_id,
                        "chunk_id": chunk_id,
                        "pmcid": pmcid,
                        "article_title": title,
                        "section": section,
                        "start_char": start,
                        "end_char": end,
                        "mention_text": ent_text,
                        "mention_type": "CONDITION",
                        "concept_id": concept_id,
                        "link_score": float(score),
                        "detector": "SCISPACY_UMLS",
                    })

mentions_raw_df = pd.DataFrame(mentions_raw)
print("Raw mentions:", len(mentions_raw_df))
mentions_raw_df.head(3)

 10%|▉         | 39/405 [02:27<10:47,  1.77s/it] 

In [ ]:
# %%
if len(mentions_raw_df) == 0:
    raise RuntimeError(
        "No linked SYMPTOM/CONDITION mentions extracted.\n"
        "Likely causes:\n"
        " - SciSpaCy model has no NER enabled, or\n"
        " - UMLS linker resources not installed, or\n"
        " - TUI filters too strict for this corpus.\n"
        "First check: doc.ents is non-empty for a test sentence; then check ent._.kb_ents."
    )

concept_df = pd.DataFrame(list(concept_rows.values()))

concept_types = (
    mentions_raw_df.groupby("concept_id")["mention_type"]
    .apply(lambda x: sorted(set(x)))
    .reset_index()
)

concept_df = concept_df.merge(concept_types, on="concept_id", how="left")
concept_df.rename(columns={"mention_type": "concept_types"}, inplace=True)

print("Concepts:", len(concept_df))
concept_df.head(3)

In [ ]:
# %%
mentions_df = (
    mentions_raw_df.sort_values("link_score", ascending=False)
    .drop_duplicates(subset=["chunk_id", "concept_id", "mention_type"])
    .reset_index(drop=True)
)

print("Mentions (deduped):", len(mentions_df))
mentions_df.head(3)

In [ ]:
# %%
SECTION_WEIGHT = {"results": 1.0, "discussion": 0.8}

chunk_meta = df.set_index("chunk_id")[["pmcid", "article_title", "section", "chunk_text"]].to_dict(orient="index")

edge_evidence_rows = []
edge_agg = defaultdict(lambda: {"support_count": 0, "weighted_score": 0.0})

by_chunk = mentions_df.groupby("chunk_id")

for chunk_id, g in tqdm(by_chunk, total=by_chunk.ngroups):
    meta = chunk_meta.get(chunk_id)
    if meta is None:
        continue

    sec = meta["section"]
    w_sec = SECTION_WEIGHT.get(sec, 0.7)

    syms = g[g["mention_type"] == "SYMPTOM"][["concept_id", "link_score"]].values.tolist()
    conds = g[g["mention_type"] == "CONDITION"][["concept_id", "link_score"]].values.tolist()

    if not syms or not conds:
        continue

    num_pairs = len(syms) * len(conds)
    damp = 1.0 / math.log(2.0 + num_pairs)

    snippet = meta["chunk_text"][:300].replace("\n", " ").strip()

    for s_id, s_score in syms:
        for c_id, c_score in conds:
            conf = float((float(s_score) + float(c_score)) / 2.0)
            evid_score = w_sec * conf * damp

            edge_evidence_rows.append({
                "symptom_concept_id": s_id,
                "condition_concept_id": c_id,
                "chunk_id": chunk_id,
                "pmcid": meta["pmcid"],
                "article_title": meta["article_title"],
                "section": sec,
                "snippet": snippet,
                "evidence_score": float(evid_score),
                "confidence": float(conf),
                "num_pairs_in_chunk": int(num_pairs),
            })

            key = (s_id, c_id)
            edge_agg[key]["support_count"] += 1
            edge_agg[key]["weighted_score"] += float(evid_score)

edge_evidence_df = pd.DataFrame(edge_evidence_rows)
print("Edge evidence rows:", len(edge_evidence_df))
edge_evidence_df.head(3)

In [ ]:
# %%
edges_sc_df = pd.DataFrame([
    {
        "symptom_concept_id": k[0],
        "condition_concept_id": k[1],
        "support_count": v["support_count"],
        "weighted_score": v["weighted_score"],
    }
    for k, v in edge_agg.items()
]).sort_values(["weighted_score", "support_count"], ascending=False).reset_index(drop=True)

print("Symptom–Condition edges:", len(edges_sc_df))
edges_sc_df.head(10)

In [ ]:
# %%
concept_lookup = concept_df.set_index("concept_id").to_dict(orient="index")

# Nodes
entity_nodes = []
for cid, row in concept_lookup.items():
    entity_nodes.append({
        "node_id": cid,
        "node_type": "ENTITY",
        "label": row.get("canonical_name"),
        "attrs": json.dumps({
            "concept_types": row.get("concept_types"),
            "cui": row.get("cui"),
            "tuis": row.get("tuis"),
            "source_vocab": row.get("source_vocab"),
            "is_grounded": row.get("is_grounded"),
        }),
    })

evidence_nodes = []
for chunk_id, meta in chunk_meta.items():
    evidence_nodes.append({
        "node_id": chunk_id,
        "node_type": "EVIDENCE",
        "label": f"{meta['section']} chunk",
        "attrs": json.dumps({
            "pmcid": meta["pmcid"],
            "section": meta["section"],
            "article_title": meta["article_title"],
        }),
    })

article_nodes = []
for pmcid, g in df.groupby("pmcid"):
    title = g["article_title"].iloc[0]
    license_code = g["license"].iloc[0] if "license" in g.columns else None
    article_nodes.append({
        "node_id": pmcid,
        "node_type": "ARTICLE",
        "label": title,
        "attrs": json.dumps({"license": license_code}),
    })

nodes_df = pd.DataFrame(entity_nodes + evidence_nodes + article_nodes)
print("Nodes:", len(nodes_df))
nodes_df.head(3)

In [ ]:
# %%
# Edges
edges_rows = []

# concept -> chunk
for _, r in mentions_df.iterrows():
    edges_rows.append({
        "src_id": r["concept_id"],
        "dst_id": r["chunk_id"],
        "edge_type": "MENTIONED_IN",
        "weight": float(r["link_score"]),
        "attrs": json.dumps({
            "mention_type": r["mention_type"],
            "mention_text": r["mention_text"],
            "start_char": int(r["start_char"]),
            "end_char": int(r["end_char"]),
            "pmcid": r["pmcid"],
            "section": r["section"],
        }),
    })

# chunk -> pmcid
for chunk_id, meta in chunk_meta.items():
    edges_rows.append({
        "src_id": chunk_id,
        "dst_id": meta["pmcid"],
        "edge_type": "IN_ARTICLE",
        "weight": 1.0,
        "attrs": json.dumps({
            "section": meta["section"],
            "article_title": meta["article_title"],
        }),
    })

# symptom -> condition
for _, r in edges_sc_df.iterrows():
    edges_rows.append({
        "src_id": r["symptom_concept_id"],
        "dst_id": r["condition_concept_id"],
        "edge_type": "CO_MENTION",
        "weight": float(r["weighted_score"]),
        "attrs": json.dumps({
            "support_count": int(r["support_count"]),
        }),
    })

edges_df = pd.DataFrame(edges_rows)
print("Edges:", len(edges_df))
edges_df.head(3)

In [ ]:
# %%
mentions_raw_df.to_parquet(OUT_MENTIONS_RAW, index=False)
concept_df.to_parquet(OUT_CONCEPTS, index=False)
mentions_df.to_parquet(OUT_MENTIONS, index=False)

edge_evidence_df.to_parquet(OUT_EDGE_EVID, index=False)
edges_sc_df.to_parquet(OUT_EDGE_SC, index=False)

nodes_df.to_parquet(OUT_NODES, index=False)
edges_df.to_parquet(OUT_EDGES, index=False)

print("Saved:")
print(" -", OUT_MENTIONS_RAW.resolve())
print(" -", OUT_CONCEPTS.resolve())
print(" -", OUT_MENTIONS.resolve())
print(" -", OUT_EDGE_EVID.resolve())
print(" -", OUT_EDGE_SC.resolve())
print(" -", OUT_NODES.resolve())
print(" -", OUT_EDGES.resolve())

In [ ]:
# %%
G = nx.MultiDiGraph()

for _, n in nodes_df.iterrows():
    G.add_node(n["node_id"], node_type=n["node_type"], label=n["label"], attrs=n["attrs"])

for _, e in edges_df.iterrows():
    G.add_edge(
        e["src_id"],
        e["dst_id"],
        edge_type=e["edge_type"],
        weight=float(e["weight"]),
        attrs=e["attrs"],
    )

with open(OUT_NX_PKL, "wb") as f:
    pickle.dump(G, f)

print("Saved NetworkX graph:", OUT_NX_PKL.resolve())
print("Graph summary:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

In [ ]:
# %%
print("Mentions raw:", len(mentions_raw_df))
print("Mentions dedup:", len(mentions_df))
print("Concepts:", len(concept_df))
print("Edge evidence:", len(edge_evidence_df))
print("Edges symptom-condition:", len(edges_sc_df))

print("\nTop edges:")
display(edges_sc_df.head(10))

print("\nSample evidence rows:")
display(edge_evidence_df.sample(min(5, len(edge_evidence_df)), random_state=42))

In [ ]:
# %%
SYMPTOM_ID = None

if SYMPTOM_ID is None:
    print("Set SYMPTOM_ID to a real concept_id to test traversal.")
else:
    top = edges_sc_df[edges_sc_df["symptom_concept_id"] == SYMPTOM_ID].head(20)
    display(top)

    if len(top) > 0:
        top_cond = top.iloc[0]["condition_concept_id"]
        ev = edge_evidence_df[
            (edge_evidence_df["symptom_concept_id"] == SYMPTOM_ID) &
            (edge_evidence_df["condition_concept_id"] == top_cond)
        ].sort_values("evidence_score", ascending=False).head(5)
        display(ev)

In [ ]:
# =========================
# NOTEBOOK 03 — GRAPH BUILD METRICS REPORT
# =========================
import pandas as pd
import numpy as np

print("=" * 90)
print("NOTEBOOK 03 METRICS — SCISPACY + UMLS + GRAPH ARTIFACTS")
print("=" * 90)

if 'df' in globals():
    print(f"Input retrieval chunks: {len(df):,}")
    print(f"Input PMCIDs: {df['pmcid'].nunique():,}")

if 'mentions_raw_df' in globals() and not mentions_raw_df.empty:
    print(f"\nRaw mentions: {len(mentions_raw_df):,}")
    print(f"Chunks with >=1 raw mention: {mentions_raw_df['chunk_id'].nunique():,}")
    print(f"Unique raw concepts: {mentions_raw_df['concept_id'].nunique():,}")

    print("\nRaw mention type distribution:")
    display(mentions_raw_df["mention_type"].value_counts(dropna=False).to_frame("count"))

    print("\nRaw linker score stats:")
    if "link_score" in mentions_raw_df.columns:
        display(mentions_raw_df["link_score"].describe())

    raw_mentions_per_chunk = mentions_raw_df.groupby("chunk_id").size()
    print("\nRaw mentions per chunk:")
    display(raw_mentions_per_chunk.describe())

if 'mentions_df' in globals() and not mentions_df.empty:
    print(f"\nDeduped mentions: {len(mentions_df):,}")
    print(f"Deduped unique chunk_ids: {mentions_df['chunk_id'].nunique():,}")
    print(f"Deduped unique concepts: {mentions_df['concept_id'].nunique():,}")

    dedupe_ratio = len(mentions_df) / len(mentions_raw_df) if 'mentions_raw_df' in globals() and len(mentions_raw_df) else np.nan
    print(f"Deduped/raw mention ratio: {dedupe_ratio:.2%}" if pd.notna(dedupe_ratio) else "Deduped/raw mention ratio: N/A")

    print("\nDeduped mention type distribution:")
    display(mentions_df["mention_type"].value_counts(dropna=False).to_frame("count"))

    dedup_mentions_per_chunk = mentions_df.groupby("chunk_id").size()
    print("\nDeduped mentions per chunk:")
    display(dedup_mentions_per_chunk.describe())

if 'concept_df' in globals() and not concept_df.empty:
    print(f"\nConcepts: {len(concept_df):,}")
    if "concept_types" in concept_df.columns:
        concept_type_counts = concept_df["concept_types"].explode().value_counts(dropna=False)
        print("\nConcept type membership:")
        display(concept_type_counts.to_frame("count"))

if 'edge_evidence_df' in globals() and not edge_evidence_df.empty:
    print(f"\nEdge evidence rows: {len(edge_evidence_df):,}")
    print(f"Distinct symptom concepts in evidence: {edge_evidence_df['symptom_concept_id'].nunique():,}")
    print(f"Distinct condition concepts in evidence: {edge_evidence_df['condition_concept_id'].nunique():,}")
    print(f"Distinct chunks contributing evidence: {edge_evidence_df['chunk_id'].nunique():,}")
    print(f"Distinct PMCIDs contributing evidence: {edge_evidence_df['pmcid'].nunique():,}")

    print("\nEvidence score stats:")
    display(edge_evidence_df["evidence_score"].describe())

    print("\nConfidence stats:")
    if "confidence" in edge_evidence_df.columns:
        display(edge_evidence_df["confidence"].describe())

    print("\nPairs-per-chunk stats:")
    if "num_pairs_in_chunk" in edge_evidence_df.columns:
        display(edge_evidence_df["num_pairs_in_chunk"].describe())

    top_symptoms = (
        edge_evidence_df.groupby("symptom_concept_id")
        .agg(
            evidence_rows=("chunk_id", "size"),
            total_evidence_score=("evidence_score", "sum"),
            distinct_pmcids=("pmcid", "nunique"),
        )
        .sort_values(["total_evidence_score", "evidence_rows"], ascending=False)
        .head(15)
    )
    print("\nTop symptom concepts by total evidence score:")
    display(top_symptoms)

    top_conditions = (
        edge_evidence_df.groupby("condition_concept_id")
        .agg(
            evidence_rows=("chunk_id", "size"),
            total_evidence_score=("evidence_score", "sum"),
            distinct_pmcids=("pmcid", "nunique"),
        )
        .sort_values(["total_evidence_score", "evidence_rows"], ascending=False)
        .head(15)
    )
    print("\nTop condition concepts by total evidence score:")
    display(top_conditions)

if 'edges_sc_df' in globals() and not edges_sc_df.empty:
    print(f"\nSymptom-condition edges: {len(edges_sc_df):,}")

    print("\nSupport count stats:")
    display(edges_sc_df["support_count"].describe())

    print("\nWeighted score stats:")
    display(edges_sc_df["weighted_score"].describe())

    if 'edge_evidence_df' in globals():
        pmcid_support = (
            edge_evidence_df.groupby(["symptom_concept_id", "condition_concept_id"])["pmcid"]
            .nunique()
            .rename("distinct_pmcids")
            .reset_index()
        )
        edge_with_diversity = edges_sc_df.merge(
            pmcid_support,
            on=["symptom_concept_id", "condition_concept_id"],
            how="left"
        )

        print("\nDistinct PMCIDs supporting each edge:")
        display(edge_with_diversity["distinct_pmcids"].describe())

        print("\nTop edges by weighted score (with article diversity):")
        display(
            edge_with_diversity
            .sort_values(["weighted_score", "support_count", "distinct_pmcids"], ascending=False)
            .head(15)
        )

if 'nodes_df' in globals() and not nodes_df.empty:
    print(f"\nGraph nodes: {len(nodes_df):,}")
    print("\nNode type distribution:")
    display(nodes_df["node_type"].value_counts(dropna=False).to_frame("count"))

if 'edges_df' in globals() and not edges_df.empty:
    print(f"\nGraph edges: {len(edges_df):,}")
    print("\nEdge type distribution:")
    display(edges_df["edge_type"].value_counts(dropna=False).to_frame("count"))

print("\nSaved artifact paths:")
for var_name in [
    "OUT_MENTIONS_RAW", "OUT_CONCEPTS", "OUT_MENTIONS",
    "OUT_EDGE_EVID", "OUT_EDGE_SC", "OUT_NODES", "OUT_EDGES"
]:
    if var_name in globals():
        print(f"{var_name}: {globals()[var_name].resolve()}")